# Instrument definition

In [ ]:
import mcstasscript as ms

trex = ms.McStas_instr(
    "trex_primary",
    author="Bing Li",
    origin="DMSC",
    input_path="./mcstas",
    output_path="./mcstas/out",
)
# ------------------- instrument parameters -------------------
trex.add_parameter(
    "L0",
    value=2.4,
    comment="wavelength in [Å] used to calculate the minimum and maximum wavelength "
    + "for the source component by adding and subtracting a set wavelength",
)

trex.add_parameter(
    "int",
    "RRM",
    value=4,
    comment="repetition rate multiplication",
)

trex.add_parameter(
    "string",
    "mode",
    value='"HF"',
    comment="HR for High Resolution, HF for High Flux",
)

# ------------ bender parameters -----------------
bender = trex.add_parameter(
    "int", "bender", value=0, comment="bender out = 0, bender in = 1"
)
b_rot = trex.add_parameter(
    "double",
    "b_rot",
    value=-0.397881,
    comment="in [deg], rotation of the cold neutron bender = -atan(0.05/7.2), full deflection since endFlat=0",
)

# Declare section

In [ ]:
trex.add_declare_var(
    "double",
    "T_offset",
    value=1.7e-3,
    comment="Time offset in [s] to shift to the center of long pulse",
)
trex.add_declare_var(
    "double",
    "f",
    value=14.0,
    comment="Source frequency in [Hz]",
)

frac = trex.add_declare_var(
    "double",
    "frac",
    value=0.3,
    comment="Used in the Source component. Defines the statistical fraction of events"
    + "emitted from the cold part of the moderator, the value is determined later in the file"
    + " depending on the moderator type that was chosen",
)

In [ ]:
trex.append_declare("""

    double lamb_1, lamb_2, d_lamb;
    double tof_BW1, tof_BW2, tof_P1, tof_P2, tof_M1, tof_M2;

    // characteristic wavelength
    double v0, v1, v2;

    // distances
    double L_BW1 = 32;
    double L_BW2 = 40;
    double L_P1 = 107.95;
    double L_P2 = 108.05;
    double L_M1 = 162.0325-0.005;
    double L_M2 = 162.0325+0.005;

    """)

trex.append_declare("""

    // BW1 Chopper
    double f_BW1;
    double delay_BW1; 

    // BW2 Chopper
    double f_BW2;
    double delay_BW2; 

    // P1 Chopper
    double f_P1;
    double delay_P1;

    // P2 Chopper
    double f_P2;
    double delay_P2;

    // M1 Chopper
    double f_M1;
    double delay_M1; 

    // M2 Chopper
    double f_M2;
    double delay_M2; 

    """)

In [ ]:
trex.append_declare("""
    // Monitor positions (geometric, taken from the guide chain) - used ONLY to
    // frame monitor time windows. These are NOT chopper phasing distances, and
    // nothing physical depends on them: Monitor_nD time limits are a histogram
    // range, not an optical element. Update if the guide geometry moves.

    double L_BW2mon = 41.984942; // BW2_monitor_ToF, 13.7 mm past the Guide_15B exit
    double L_Pmon   = 111.098;   // P_monitor_ToF_cm2, exit of Guide_33
    double L_Smon   = 163.800;   // sample monitor sits AT Sample_pos

    // monitor definitions
    
    int lbin;
    int ebin;
    char setBM1[256];
""")


# the chopper train the masked source reads: the same six discs as the components below
trex.append_declare("""
    chopper_parameters *train;
    double *train_as_doubles;
    double beam_center, aperture;

    // the beam window as the discs see it, from their spindles
    double disc_radius = 0.35;
    """)

# Initialize section

In [ ]:
trex.append_initialize("""
    // frequencies

    f_BW1 = f;
    f_BW2 = f_BW1;
    f_P1 = RRM*f*0.75;
    f_P2 = -f_P1;
    f_M1 = RRM*f;
    f_M2 = f_M1;

    // Resolution Settings
    if (strcmp(mode, "HF") == 0){
        for (int i = 0; i < 8; i++) {
            slit_edges_PS_Chopper_1[i] += 55;
            slit_edges_PS_Chopper_2[i] += 55;
        }
        for (int i = 0; i < 4; i++) {
            slit_edges_MC_Chopper_1[i] += 175;
            slit_edges_MC_Chopper_2[i] += 175;
        }
    } else if (strcmp(mode, "HR") != 0) {
        printf("trex_primary: need valid resolution option, 'HR' or 'HF'\\n");
        exit(-1);
    }

    // characteristic wavelength 
    d_lamb = 3956.0 / (f * RRM * 162);
    lamb_1 = L0-d_lamb*0.5;
    lamb_2 = L0+d_lamb*0.5;

    v0 = 2*PI*K2V/L0; // speed center neutron
    v1 = 2*PI*K2V/lamb_1;   // fastest neutron
	v2 = 2*PI*K2V/lamb_2;   // slowest neutron
    


    // time of flight and chopper phases
    tof_BW1 = T_offset + L_BW1/v0;
    tof_BW2 = T_offset + L_BW2/v0;
    tof_P1 = T_offset + L_P1/v0;
    tof_P2 = T_offset + L_P2/v0;
    tof_M1 = T_offset + L_M1/v0;
    tof_M2 = T_offset + L_M2/v0;

    //time delay
    delay_BW1 = tof_BW1; 
    delay_BW2 = tof_BW2; 
    delay_P1 = tof_P1; 
    delay_P2 = tof_P2;
    delay_M1 = tof_M1;
    delay_M2 = tof_M2;
    """)

In [ ]:
# The train is built after the frequencies, delays and HF/HR edge shifts above, since it
# copies their values. Order follows the beam: BW1, BW2, PS1, PS2, MC1, MC2.
trex.append_initialize("""
    train = (chopper_parameters *) calloc(6, sizeof(chopper_parameters));
    if (train == NULL) {
        printf("trex_primary: out of memory building the chopper train\\n");
        exit(-1);
    }
    

    // angle the beam window subtends about the spindle, per disc; its inner corners lie
    // yheight inside the rim at the edge of the window
    // h2, w2 of guide 13a, 15a, 32 and 49
    double beam_width[6] = {0.06, 0.06, 0.06, 0.06, 0.022433, 0.022433};
    double beam_height[6] = {0.083061, 0.084983, 0.085, 0.085, 0.034987, 0.034987};
    double disc_beam[6] = {0, 0, 180, 180, 180, 0};
    double disc_yheight[6] = {0.085, 0.085, 0.095, 0.095, 0.045, 0.045};
    
    double disc_speed[6]   = {f_BW1, f_BW2, f_P1, f_P2, f_M1, f_M2};
    double disc_delay[6]   = {delay_BW1, delay_BW2, delay_P1, delay_P2, delay_M1, delay_M2};
    double disc_path[6]    = {L_BW1, L_BW2, L_P1, L_P2, L_M1, L_M2};
    int    disc_nedges[6]  = {2, 2, 8, 8, 4, 4};
    double *disc_edges[6]  = {slit_edges_BW_Chopper_1, slit_edges_BW_Chopper_2,
                              slit_edges_PS_Chopper_1, slit_edges_PS_Chopper_2,
                              slit_edges_MC_Chopper_1, slit_edges_MC_Chopper_2};

    
    for (int i = 0; i < 6; i++) {
        beam_center = disc_radius - disc_yheight[i] / 2.0;
        aperture = 2 * RAD2DEG * atan2(beam_width[i] / 2.0, beam_center - beam_height[i] / 2.0);
        train[i] = (chopper_parameters){disc_speed[i], disc_delay[i], disc_beam[i], disc_nedges[i],
                                        disc_edges[i], disc_path[i], aperture};
    }

    // a component parameter carries only `double *`, so the train travels as one
    train_as_doubles = (double *) train;
    """)

In [ ]:
trex.append_initialize("""
// Monitors

lbin = round((lamb_2-lamb_1)/0.002);    // number of bins for TOF monitors

sprintf(setBM1, "time limits [%g %g] bins = [%i]", T_offset + L_BW2mon/v1, T_offset + L_BW2mon/v2, lbin);""")

In [ ]:
trex.show_variables()

# Trace section

In [ ]:
from mctrex.mcstas_components import add_components_to
from mctrex.params.source import masked_source
from mctrex.trex_primary import components

# swap the plain ESS_butterfly for the masked one, keeping its place in the order
components = {k: (masked_source if k == "source" else v) for k, v in components.items()}

components["source"].cold_frac = frac
components["source"].Lmin = 1.0
components["source"].Lmax = 5.0
components["source"].choppers = "train_as_doubles"
components["source"].chopper_count = 6
components["source"].resample = 1  # redraw excluded rays inside the mask instead of absorbing them

components["bender"].ROTATED = [0, b_rot, 0]

components["BW_Chopper_1"].nu = "f_BW1"
components["BW_Chopper_1"].delay = "delay_BW1"

components["BW_Chopper_2"].nu = "f_BW2"
components["BW_Chopper_2"].delay = "delay_BW2"

components["PS_Chopper_1"].nu = "f_P1"

components["PS_Chopper_1"].delay = "delay_P1"
components["PS_Chopper_2"].nu = "f_P2"
components["PS_Chopper_2"].delay = "delay_P2"

components["MC_Chopper_1"].nu = "f_M1"
components["MC_Chopper_1"].delay = "delay_M1"

components["MC_Chopper_2"].nu = "f_M2"
components["MC_Chopper_2"].delay = "delay_M2"

trex = add_components_to(trex, components)

trex.show_parameters()

In [ ]:
trex.show_components()

In [ ]:
trex.backengine()

In [ ]:
#%matplotlib widget
# %matplotlib inline
#trex.show_instrument(backend="matplotlib_2d",projection="xy")
# trex.show_instrument(backend="pythreejs")

In [ ]:
# %matplotlib widget
# trex.show_instrument(backend="matplotlib_2d",projection="zx")


In [ ]:
%matplotlib inline
trex.set_parameters(bender=0)

trex.show_diagram(analysis=True)
trex.show_diagram(variable="l", limits=[1, 4])

# Diagnostics

In [ ]:
diag = ms.Diagnostics(trex)

diag.settings(ncount=1e8, suppress_output=True, mpi="auto")
diag.show_settings()

In [ ]:
diag.clear_points()
diag.add_point(after="source")
diag.add_point(after="Guide_15A")
diag.add_point(after="BW_Chopper_2")
diag.add_point(after="Guide_32")
diag.add_point(after="PS_Chopper_2")
diag.add_point(after="Guide_49")
diag.add_point(after="MC_Chopper_2")
diag.add_point(after="slitMC")

In [ ]:
diag.set_parameters(bender=0)
diag.show_parameters()

In [ ]:
diag.run()

In [ ]:
diag.clear_views()
diag.add_view("x", "y", bins=100,same_scale=True)
# diag.add_view("vx", "vy", bins=200)
# diag.add_view("dx", "dy", bins=100)
# diag.add_view("l", bins=100)
# diag.add_view("t", bins=100)
# diag.add_view("l", "y", bins=100, same_scale=True)
diag.add_view("t", "l", bins=[2000,400], same_scale=True, top_lim=4, bottom_lim=1,left_lim=0.0,right_lim=0.15)
diag.add_view("t", bins=2000,  same_scale=True,left_lim=0,right_lim=0.2)
diag.add_view("l", bins=400,  same_scale=True,left_lim=1,right_lim=4)


In [ ]:
%matplotlib widget
# %matplotlib inline
diag.plot()